In [1]:

from pathlib import Path
from ultralytics import YOLO, RTDETR
import yaml

# Project root
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Locate test images from dataset YAML
dataset = yaml.safe_load(
    (ROOT / "data/beach_bird.yaml").read_text()
)

data_root = Path(dataset["path"])
if not data_root.is_absolute():
    data_root = ROOT / data_root

test_dir = data_root / dataset["test"]
assert test_dir.is_dir(), f"Test folder not found: {test_dir}"

# Output directory
output_root = ROOT / "results"

# Final trained models
experiments = [
    (
        "yolo11n_test",
        YOLO,
        ROOT / "runs/train/yolo11n_baseline_v1/weights/best.pt"
    ),
    (
        "rtdetr_l_test",
        RTDETR,
        ROOT / "runs/train/rtdetr_l_stable_v1-2/weights/best.pt"
    )
]

for name, model_class, weights in experiments:
    assert weights.is_file(), f"Missing weights: {weights}"

    print(f"\nRunning: {name}")
    model = model_class(str(weights))

    count = 0

    predictions = model.predict(
        source=str(test_dir),
        imgsz=640,
        batch=2,
        device="mps",
        conf=0.25,
        save=True,
        save_txt=True,
        save_conf=True,
        show_labels=True,
        show_conf=True,
        line_width=2,
        project=str(output_root),
        name=name,
        exist_ok=False,
        stream=True,
        verbose=False
    )

    for _ in predictions:
        count += 1

    print(f"Processed images: {count}")
    print(f"Saved to: {model.predictor.save_dir}")

print("\nAll predictions completed.")



Running: yolo11n_test
Results saved to /Users/luzhang/COMP9517/comp9517-bird-detection/results/yolo11n_test
217 labels saved to /Users/luzhang/COMP9517/comp9517-bird-detection/results/yolo11n_test/labels
Processed images: 240
Saved to: /Users/luzhang/COMP9517/comp9517-bird-detection/results/yolo11n_test

Running: rtdetr_l_test
Results saved to /Users/luzhang/COMP9517/comp9517-bird-detection/results/rtdetr_l_test
216 labels saved to /Users/luzhang/COMP9517/comp9517-bird-detection/results/rtdetr_l_test/labels
Processed images: 240
Saved to: /Users/luzhang/COMP9517/comp9517-bird-detection/results/rtdetr_l_test

All predictions completed.


In [2]:

from pathlib import Path
import numpy as np
import pandas as pd
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Dataset paths
ds = yaml.safe_load(
    (ROOT / "data/beach_bird.yaml").read_text()
)

data_root = Path(ds["path"])
if not data_root.is_absolute():
    data_root = ROOT / data_root

image_dir = data_root / ds["test"]
gt_dir = data_root / "labels/test"

image_files = sorted(
    p for p in image_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

assert len(image_files) == 240
assert len({p.stem for p in image_files}) == 240


def read_labels(path, prediction=False):
    if not path.exists():
        return []

    records = []
    for line in path.read_text().splitlines():
        if not line.strip():
            continue

        values = list(map(float, line.split()))

        cls = int(values[0])
        x, y, w, h = values[1:5]

        box = [
            x - w / 2, y - h / 2,
            x + w / 2, y + h / 2
        ]

        conf = values[5] if prediction else 1.0
        records.append((cls, box, conf))

    return records


def calculate_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    intersection = max(0, x2-x1) * max(0, y2-y1)

    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])

    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def evaluate_predictions(folder):
    pred_dir = ROOT / "results" / folder / "labels"
    assert pred_dir.is_dir(), f"Not found: {pred_dir}"

    tp = fp = fn = 0

    for image_path in image_files:
        stem = image_path.stem

        gt = read_labels(gt_dir / f"{stem}.txt")
        pred = read_labels(
            pred_dir / f"{stem}.txt",
            prediction=True
        )

        # Same visualization confidence threshold
        pred = [p for p in pred if p[2] >= 0.25]
        pred.sort(key=lambda p: p[2], reverse=True)

        matched = set()

        for pred_cls, pred_box, confidence in pred:
            best_iou = 0.0
            best_idx = -1

            for i, (gt_cls, gt_box, _) in enumerate(gt):
                if i in matched or pred_cls != gt_cls:
                    continue

                iou = calculate_iou(pred_box, gt_box)

                if iou > best_iou:
                    best_iou = iou
                    best_idx = i

            if best_iou >= 0.50:
                tp += 1
                matched.add(best_idx)
            else:
                fp += 1

        fn += len(gt) - len(matched)

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision@0.25": tp / (tp + fp) if tp + fp else 0,
        "Recall@0.25": tp / (tp + fn) if tp + fn else 0,
        "FP/Image": fp / len(image_files)
    }


results = pd.DataFrame({
    "YOLO11n": evaluate_predictions("yolo11n_test"),
    "RT-DETR-L": evaluate_predictions("rtdetr_l_test")
}).T

display(results.round(4))

# Save analysis
output_path = ROOT / "results/fixed_threshold_comparison.csv"
results.to_csv(output_path)

print("Saved:", output_path)


,TP,FP,FN,Precision@0.25,Recall@0.25,FP/Image
YOLO11n,1276.0,411.0,242.0,0.7564,0.8406,1.7125
RT-DETR-L,1412.0,954.0,106.0,0.5968,0.9302,3.9750


Saved: /Users/luzhang/COMP9517/comp9517-bird-detection/results/fixed_threshold_comparison.csv
